In [2]:
import torch

from torch_geometric.nn import HANConv
from torch_geometric.datasets import IMDB, DBLP
from torch_geometric.transforms import AddMetaPaths

In [3]:
dblp_data = DBLP(root='../data/dblp/')

Extracting ..\data\dblp\raw\DBLP_processed.zip
Processing...
Done!


## Basic Sub

In [50]:
from typing import Dict, List, Optional, Tuple, Union

import torch
import torch.nn.functional as F
from torch import Tensor, nn

from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.dense import Linear
from torch_geometric.nn.inits import glorot, reset
from torch_geometric.typing import Adj, EdgeType, Metadata, NodeType, OptTensor
from torch_geometric.utils import softmax


def group(
    xs: List[Tensor],
    q: nn.Parameter,
    k_lin: nn.Module,
) -> Tuple[OptTensor, OptTensor]:
    if len(xs) == 0:
        return None, None
    else:
        num_edge_types = len(xs)
        out = torch.stack(xs)  # [num_edge_types, num_nodes, out_channels]
        if out.numel() == 0:
            return out.view(0, out.size(-1)), None
        attn_score = (q * torch.tanh(k_lin(out)).mean(1)).sum(-1)
        attn = F.softmax(attn_score, dim=0)
        out = torch.sum(attn.view(num_edge_types, 1, -1) * out, dim=0)
        return out, attn


class HANConv(MessagePassing):
    def __init__(
        self,
        in_channels: Union[int, Dict[str, int]],
        out_channels: int,
        metadata: Metadata,
        heads: int = 1,
        negative_slope=0.2,
        dropout: float = 0.0,
        **kwargs,
    ):
        super().__init__(aggr='add', node_dim=0, **kwargs)

        if not isinstance(in_channels, dict):
            in_channels = {node_type: in_channels for node_type in metadata[0]}

        self.heads = heads
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.negative_slope = negative_slope
        self.metadata = metadata
        self.dropout = dropout

        self.k_lin = nn.Linear(out_channels, out_channels)
        self.q = nn.Parameter(torch.empty(1, out_channels))

        self.proj = nn.ModuleDict()
        for node_type, in_channels in self.in_channels.items():
            self.proj[node_type] = Linear(in_channels, out_channels)

        self.lin_src = nn.ParameterDict()
        self.lin_dst = nn.ParameterDict()
        dim = out_channels // heads
        for edge_type in metadata[1]:
            edge_type_str = '__'.join(edge_type)
            self.lin_src[edge_type_str] = nn.Parameter(torch.empty(1, heads, dim))
            self.lin_dst[edge_type_str] = nn.Parameter(torch.empty(1, heads, dim))

        # 🆕 Add subpath embeddings for each edge type (metapath)
        self.subpath_embeds = nn.ParameterDict()
        for edge_type in metadata[1]:
            edge_type_str = '__'.join(edge_type)
            self.subpath_embeds[edge_type_str] = nn.Parameter(torch.empty(1, out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        super().reset_parameters()
        reset(self.proj)
        glorot(self.lin_src)
        glorot(self.lin_dst)
        glorot(self.q)
        self.k_lin.reset_parameters()
        glorot(self.subpath_embeds)

    def forward(
        self,
        x_dict: Dict[NodeType, Tensor],
        edge_index_dict: Dict[EdgeType, Adj],
        return_semantic_attention_weights: bool = False,
    ) -> Union[Dict[NodeType, OptTensor], Tuple[Dict[NodeType, OptTensor], Dict[NodeType, OptTensor]]]:
        H, D = self.heads, self.out_channels // self.heads
        x_node_dict, out_dict = {}, {}

        for node_type, x in x_dict.items():
            x_node_dict[node_type] = self.proj[node_type](x).view(-1, H, D)
            out_dict[node_type] = []

        for edge_type, edge_index in edge_index_dict.items():
            src_type, _, dst_type = edge_type
            edge_type_str = '__'.join(edge_type)

            lin_src = self.lin_src[edge_type_str]
            lin_dst = self.lin_dst[edge_type_str]
            x_src = x_node_dict[src_type]
            x_dst = x_node_dict[dst_type]
            alpha_src = (x_src * lin_src).sum(dim=-1)
            alpha_dst = (x_dst * lin_dst).sum(dim=-1)

            # propagate_type: (x: PairTensor, alpha: PairTensor)
            out = self.propagate(
                edge_index,
                x=(x_src, x_dst),
                alpha=(alpha_src, alpha_dst),
                edge_type=edge_type_str  # pass str name to use in message()
            )
            out = F.relu(out)
            out_dict[dst_type].append(out)

        semantic_attn_dict = {}
        for node_type, outs in out_dict.items():
            out, attn = group(outs, self.q, self.k_lin)
            out_dict[node_type] = out
            semantic_attn_dict[node_type] = attn

        if return_semantic_attention_weights:
            return out_dict, semantic_attn_dict

        return out_dict

    def message(self, x_j: Tensor, alpha_i: Tensor, alpha_j: Tensor,
                index: Tensor, ptr: Optional[Tensor], size_i: Optional[int],
                edge_type: Optional[str] = None) -> Tensor:
        alpha = alpha_j + alpha_i
        alpha = F.leaky_relu(alpha, self.negative_slope)
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        if edge_type is not None:
            sub_embed = self.subpath_embeds[edge_type]  # [1, out_channels]
            sub_embed = sub_embed.view(1, self.heads, -1)  # [1, heads, dim]
            sub_embed = sub_embed.expand(x_j.size(0), -1, -1)  # [N, heads, dim]
            x_j = x_j + sub_embed  # Inject subpath embedding

        out = x_j * alpha.view(-1, self.heads, 1)
        return out.view(-1, self.out_channels)

    def __repr__(self) -> str:
        return f'{self.__class__.__name__}({self.out_channels}, heads={self.heads})'


In [ ]:
from typing import Dict, List, Optional, Tuple, Union

import torch
import torch.nn.functional as F
from torch import Tensor, nn

from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.dense import Linear
from torch_geometric.nn.inits import glorot, reset
from torch_geometric.typing import Adj, EdgeType, Metadata, NodeType, OptTensor
from torch_geometric.utils import softmax


def group(
    xs: List[Tensor],
    q: nn.Parameter,
    k_lin: nn.Module,
) -> Tuple[OptTensor, OptTensor]:
    if len(xs) == 0:
        return None, None
    else:
        num_edge_types = len(xs)
        out = torch.stack(xs)  # [num_edge_types, num_nodes, out_channels]
        if out.numel() == 0:
            return out.view(0, out.size(-1)), None
        attn_score = (q * torch.tanh(k_lin(out)).mean(1)).sum(-1)
        attn = F.softmax(attn_score, dim=0)
        out = torch.sum(attn.view(num_edge_types, 1, -1) * out, dim=0)
        return out, attn


class HANConv(MessagePassing):
    def __init__(
        self,
        in_channels: Union[int, Dict[str, int]],
        out_channels: int,
        metadata: Metadata,
        heads: int = 1,
        negative_slope=0.2,
        dropout: float = 0.0,
        **kwargs,
    ):
        super().__init__(aggr='add', node_dim=0, **kwargs)

        if not isinstance(in_channels, dict):
            in_channels = {node_type: in_channels for node_type in metadata[0]}

        self.heads = heads
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.negative_slope = negative_slope
        self.metadata = metadata
        self.dropout = dropout

        self.k_lin = nn.Linear(out_channels, out_channels)
        self.q = nn.Parameter(torch.empty(1, out_channels))

        self.proj = nn.ModuleDict()
        for node_type, in_channels in self.in_channels.items():
            self.proj[node_type] = Linear(in_channels, out_channels)

        self.lin_src = nn.ParameterDict()
        self.lin_dst = nn.ParameterDict()
        dim = out_channels // heads
        for edge_type in metadata[1]:
            edge_type_str = '__'.join(edge_type)
            self.lin_src[edge_type_str] = nn.Parameter(torch.empty(1, heads, dim))
            self.lin_dst[edge_type_str] = nn.Parameter(torch.empty(1, heads, dim))

        # 🆕 Add subpath embeddings for each edge type (metapath)
        self.subpath_embeds = nn.ParameterDict()
        for edge_type in metadata[1]:
            edge_type_str = '__'.join(edge_type)
            self.subpath_embeds[edge_type_str] = nn.Parameter(torch.empty(1, out_channels))

        self.reset_parameters()

    def reset_parameters(self):
        super().reset_parameters()
        reset(self.proj)
        glorot(self.lin_src)
        glorot(self.lin_dst)
        glorot(self.q)
        self.k_lin.reset_parameters()
        glorot(self.subpath_embeds)

    def forward(
        self,
        x_dict: Dict[NodeType, Tensor],
        edge_index_dict: Dict[EdgeType, Adj],
        return_semantic_attention_weights: bool = False,
    ) -> Union[Dict[NodeType, OptTensor], Tuple[Dict[NodeType, OptTensor], Dict[NodeType, OptTensor]]]:
        H, D = self.heads, self.out_channels // self.heads
        x_node_dict, out_dict = {}, {}

        for node_type, x in x_dict.items():
            x_node_dict[node_type] = self.proj[node_type](x).view(-1, H, D)
            out_dict[node_type] = []

        for edge_type, edge_index in edge_index_dict.items():
            src_type, _, dst_type = edge_type
            edge_type_str = '__'.join(edge_type)

            lin_src = self.lin_src[edge_type_str]
            lin_dst = self.lin_dst[edge_type_str]
            x_src = x_node_dict[src_type]
            x_dst = x_node_dict[dst_type]
            alpha_src = (x_src * lin_src).sum(dim=-1)
            alpha_dst = (x_dst * lin_dst).sum(dim=-1)

            # propagate_type: (x: PairTensor, alpha: PairTensor)
            out = self.propagate(
                edge_index,
                x=(x_src, x_dst),
                alpha=(alpha_src, alpha_dst),
                edge_type=edge_type_str  # pass str name to use in message()
            )
            out = F.relu(out)
            out_dict[dst_type].append(out)

        semantic_attn_dict = {}
        for node_type, outs in out_dict.items():
            out, attn = group(outs, self.q, self.k_lin)
            out_dict[node_type] = out
            semantic_attn_dict[node_type] = attn

        if return_semantic_attention_weights:
            return out_dict, semantic_attn_dict

        return out_dict

    def message(self, x_j: Tensor, alpha_i: Tensor, alpha_j: Tensor,
                index: Tensor, ptr: Optional[Tensor], size_i: Optional[int],
                edge_type: Optional[str] = None) -> Tensor:
        alpha = alpha_j + alpha_i
        alpha = F.leaky_relu(alpha, self.negative_slope)
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        if edge_type is not None:
            sub_embed = self.subpath_embeds[edge_type]  # [1, out_channels]
            sub_embed = sub_embed.view(1, self.heads, -1)  # [1, heads, dim]
            sub_embed = sub_embed.expand(x_j.size(0), -1, -1)  # [N, heads, dim]
            x_j = x_j + sub_embed  # Inject subpath embedding

        out = x_j * alpha.view(-1, self.heads, 1)
        return out.view(-1, self.out_channels)

    def __repr__(self) -> str:
        return f'{self.__class__.__name__}({self.out_channels}, heads={self.heads})'


## Enhanced Sub

In [30]:
from typing import Dict, List, Optional, Tuple, Union

import torch
import torch.nn.functional as F
from torch import Tensor, nn

from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.dense import Linear
from torch_geometric.nn.inits import glorot, reset
from torch_geometric.typing import Adj, EdgeType, Metadata, NodeType, OptTensor
from torch_geometric.utils import softmax


def group(xs: List[Tensor], q: nn.Parameter, k_lin: nn.Module) -> Tuple[OptTensor, OptTensor]:
    if len(xs) == 0:
        return None, None
    else:
        out = torch.stack(xs)  # [num_edge_types, num_nodes, out_channels]
        if out.numel() == 0:
            return out.view(0, out.size(-1)), None
        attn_score = (q * torch.tanh(k_lin(out)).mean(1)).sum(-1)
        attn = F.softmax(attn_score, dim=0)
        out = torch.sum(attn.view(len(xs), 1, -1) * out, dim=0)
        return out, attn


class HANConv(MessagePassing):
    def __init__(
        self,
        in_channels: Union[int, Dict[str, int]],
        out_channels: int,
        metadata: Metadata,
        heads: int = 1,
        negative_slope=0.2,
        dropout: float = 0.2,
        subpath_mode: str = 'add',  # 'add' or 'concat'
        use_layernorm: bool = True,
        **kwargs,
    ):
        super().__init__(aggr='add', node_dim=0, **kwargs)

        if not isinstance(in_channels, dict):
            in_channels = {node_type: in_channels for node_type in metadata[0]}

        self.heads = heads
        self.out_channels = out_channels
        self.negative_slope = negative_slope
        self.dropout = dropout
        self.metadata = metadata
        self.subpath_mode = subpath_mode
        self.use_layernorm = use_layernorm

        self.k_lin = nn.Linear(out_channels, out_channels)
        self.q = nn.Parameter(torch.empty(1, out_channels))

        self.proj = nn.ModuleDict()
        for node_type, in_channels in in_channels.items():
            self.proj[node_type] = Linear(in_channels, out_channels)

        self.lin_src = nn.ParameterDict()
        self.lin_dst = nn.ParameterDict()
        dim = out_channels // heads
        for edge_type in metadata[1]:
            etype = '__'.join(edge_type)
            self.lin_src[etype] = nn.Parameter(torch.empty(1, heads, dim))
            self.lin_dst[etype] = nn.Parameter(torch.empty(1, heads, dim))

        # Subpath embeddings (one per edge type/metapath)
        self.subpath_embeds = nn.ParameterDict()
        for edge_type in metadata[1]:
            etype = '__'.join(edge_type)
            self.subpath_embeds[etype] = nn.Parameter(torch.empty(1, out_channels))

        if use_layernorm:
            self.norm = nn.LayerNorm(out_channels)

        self.lin_out = nn.Sequential(
            nn.Linear(out_channels, out_channels),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.reset_parameters()

    def reset_parameters(self):
        reset(self.proj)
        glorot(self.lin_src)
        glorot(self.lin_dst)
        glorot(self.q)
        glorot(self.subpath_embeds)
        self.k_lin.reset_parameters()
        if self.use_layernorm:
            self.norm.reset_parameters()
        for layer in self.lin_out:
            if hasattr(layer, 'reset_parameters'):
                layer.reset_parameters()

    def forward(
        self,
        x_dict: Dict[NodeType, Tensor],
        edge_index_dict: Dict[EdgeType, Adj],
        return_semantic_attention_weights: bool = False,
    ) -> Union[Dict[NodeType, OptTensor], Tuple[Dict[NodeType, OptTensor], Dict[NodeType, OptTensor]]]:
        H, D = self.heads, self.out_channels // self.heads
        x_node_dict, out_dict = {}, {}

        for node_type, x in x_dict.items():
            x_node_dict[node_type] = self.proj[node_type](x).view(-1, H, D)
            out_dict[node_type] = []

        for edge_type, edge_index in edge_index_dict.items():
            src_type, _, dst_type = edge_type
            etype = '__'.join(edge_type)

            x_src = x_node_dict[src_type]
            x_dst = x_node_dict[dst_type]
            alpha_src = (x_src * self.lin_src[etype]).sum(dim=-1)
            alpha_dst = (x_dst * self.lin_dst[etype]).sum(dim=-1)

            out = self.propagate(
                edge_index,
                x=(x_src, x_dst),
                alpha=(alpha_src, alpha_dst),
                edge_type=etype
            )
            out = F.relu(out)
            out_dict[dst_type].append(out)

        semantic_attn_dict = {}
        for node_type, outs in out_dict.items():
            out, attn = group(outs, self.q, self.k_lin)
            if out is not None:
                if self.use_layernorm:
                    out = self.norm(out)
                out = self.lin_out(out)
            out_dict[node_type] = out
            semantic_attn_dict[node_type] = attn

        if return_semantic_attention_weights:
            return out_dict, semantic_attn_dict

        return out_dict

    def message(
        self,
        x_j: Tensor,
        alpha_i: Tensor,
        alpha_j: Tensor,
        index: Tensor,
        ptr: Optional[Tensor],
        size_i: Optional[int],
        edge_type: Optional[str] = None
    ) -> Tensor:
        alpha = F.leaky_relu(alpha_j + alpha_i, self.negative_slope)
        alpha = softmax(alpha, index, ptr, size_i)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        # Inject subpath embedding
        if edge_type is not None:
            sub_embed = self.subpath_embeds[edge_type]  # [1, out_channels]
            sub_embed = sub_embed.view(1, self.heads, -1).expand(x_j.size(0), -1, -1)

            if self.subpath_mode == 'add':
                x_j = x_j + sub_embed
            elif self.subpath_mode == 'concat':
                raise NotImplementedError("Concatenation mode not yet supported in projection")
            else:
                raise ValueError(f"Unsupported subpath_mode: {self.subpath_mode}")

        out = x_j * alpha.view(-1, self.heads, 1)
        return out.view(-1, self.out_channels)

    def __repr__(self) -> str:
        return f'{self.__class__.__name__}({self.out_channels}, heads={self.heads})'

## Training

In [54]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.datasets import IMDB
from torch_geometric.nn import HANConv
from torch_geometric.transforms import AddMetaPaths

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# === Load Dataset with MetaPaths === #
# AddMetaPaths adds new edges to simulate metapath relations
transform = AddMetaPaths(
    metapaths=[
        [('movie', 'actor'), ('actor', 'movie')],  # MAM
        [('movie', 'director'), ('director', 'movie')],  # MDM
        # [('movie', 'actor'), ('actor', 'movie'), ('movie', 'director'), ('director', 'movie')],  # MAMDM
    	# [('movie', 'director'), ('director', 'movie'), ('movie', 'actor'), ('actor', 'movie')],  # MDMW
    ], # type: ignore
    drop_orig_edges=True,  # only keep metapath-based edges
    keep_same_node_type=True
)

# Apply transform on load
dataset = IMDB(root='/tmp/imdb', transform=transform)
data = dataset[0].to(device)

print("Node types:", data.node_types)
print("Edge types:", data.edge_types)  # Now includes ('movie', 'MAM', 'movie'), etc.

# === Dummy Features if not already provided === #
for node_type in data.node_types:
    if 'x' not in data[node_type]:
        num_nodes = data[node_type].num_nodes
        data[node_type].x = torch.eye(num_nodes, device=device)

# === HAN Model === #
class HAN(torch.nn.Module):
    def __init__(self, metadata, hidden_channels=64, out_channels=3):
        super().__init__()
        self.han_conv = HANConv(
            in_channels=dataset.num_features,
            out_channels=hidden_channels,
            metadata=metadata,
            heads=8
        )
        self.lin = Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict):
        x = self.han_conv(x_dict, edge_index_dict)
        x = F.elu(x['movie'])
        return self.lin(x)

# === Model, Loss, Optimizer === #
model = HAN(
    metadata=data.metadata(),
    hidden_channels=64,
    out_channels=3  # 3 movie classes: Action, Comedy, Drama
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

# === Labels and Train/Test Split === #


Node types: ['movie', 'director', 'actor']
Edge types: [('movie', 'metapath_0', 'movie'), ('movie', 'metapath_1', 'movie')]


c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\transforms\add_metapaths.py:111: UserWarning: 'drop_orig_edges' is dprecated. Use 'drop_orig_edge_types' instead
  warnings.warn("'drop_orig_edges' is dprecated. Use "


In [55]:
from sklearn.metrics import f1_score

target = data['movie'].y
num_nodes = target.size(0)
perm = torch.randperm(num_nodes, device=device)
train_idx = perm[:int(0.8 * num_nodes)]
test_idx = perm[int(0.8 * num_nodes):]

# === Train & Test === #
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x_dict, data.edge_index_dict)
    loss = criterion(out[train_idx], target[train_idx])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test():
    model.eval()
    out = model(data.x_dict, data.edge_index_dict)
    pred = out.argmax(dim=1)
    y_true = target[test_idx].cpu().numpy()
    y_pred = pred[test_idx].cpu().numpy()
    micro_f1 = f1_score(y_true, y_pred, average='micro')
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    # print(f"Micro-F1: {micro_f1:.4f} | Macro-F1: {macro_f1:.4f}")
    # acc = (pred[test_idx] == target[test_idx]).float().mean()
    return micro_f1, macro_f1

# === Run Training Loop === #
for epoch in range(1, 101):
    loss = train()
    micro_f1, macro_f1 = test()
    print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Micro-F1: {micro_f1:.4f} | Macro-F1: {macro_f1:.4f}")

Epoch 001 | Loss: 1.1053 | Micro-F1: 0.5023 | Macro-F1: 0.4012
Epoch 002 | Loss: 1.0893 | Micro-F1: 0.4591 | Macro-F1: 0.3525
Epoch 003 | Loss: 1.0702 | Micro-F1: 0.5058 | Macro-F1: 0.4367
Epoch 004 | Loss: 1.0476 | Micro-F1: 0.5526 | Macro-F1: 0.5010
Epoch 005 | Loss: 1.0228 | Micro-F1: 0.5853 | Macro-F1: 0.5402
Epoch 006 | Loss: 0.9965 | Micro-F1: 0.6051 | Macro-F1: 0.5652
Epoch 007 | Loss: 0.9688 | Micro-F1: 0.6238 | Macro-F1: 0.5901
Epoch 008 | Loss: 0.9400 | Micro-F1: 0.6308 | Macro-F1: 0.6029
Epoch 009 | Loss: 0.9104 | Micro-F1: 0.6402 | Macro-F1: 0.6164
Epoch 010 | Loss: 0.8803 | Micro-F1: 0.6519 | Macro-F1: 0.6328
Epoch 011 | Loss: 0.8499 | Micro-F1: 0.6565 | Macro-F1: 0.6396
Epoch 012 | Loss: 0.8196 | Micro-F1: 0.6612 | Macro-F1: 0.6465
Epoch 013 | Loss: 0.7896 | Micro-F1: 0.6682 | Macro-F1: 0.6575
Epoch 014 | Loss: 0.7603 | Micro-F1: 0.6717 | Macro-F1: 0.6635
Epoch 015 | Loss: 0.7317 | Micro-F1: 0.6752 | Macro-F1: 0.6693
Epoch 016 | Loss: 0.7042 | Micro-F1: 0.6776 | Macro-F1:

In [ ]:
# from typing import Dict, List, Optional, Tuple, Union

# import torch
# import torch.nn.functional as F
# from torch import Tensor, nn

# from torch_geometric.nn.conv import MessagePassing
# from torch_geometric.nn.dense import Linear
# from torch_geometric.nn.inits import glorot, reset
# from torch_geometric.typing import PairTensor  # noqa
# from torch_geometric.typing import Adj, EdgeType, Metadata, NodeType, OptTensor
# from torch_geometric.utils import softmax


# def group(
#     xs: List[Tensor],
#     q: nn.Parameter,
#     k_lin: nn.Module,
# ) -> Tuple[OptTensor, OptTensor]:

#     if len(xs) == 0:
#         return None, None
#     else:
#         num_edge_types = len(xs)
#         out = torch.stack(xs)
#         if out.numel() == 0:
#             return out.view(0, out.size(-1)), None
#         attn_score = (q * torch.tanh(k_lin(out)).mean(1)).sum(-1)
#         attn = F.softmax(attn_score, dim=0)
#         out = torch.sum(attn.view(num_edge_types, 1, -1) * out, dim=0)
#         return out, attn


# class HANConv(MessagePassing):
#     r"""The Heterogenous Graph Attention Operator from the
#     `"Heterogenous Graph Attention Network"
#     <https://arxiv.org/abs/1903.07293>`_ paper.

#     .. note::

#         For an example of using HANConv, see `examples/hetero/han_imdb.py
#         <https://github.com/pyg-team/pytorch_geometric/blob/master/examples/
#         hetero/han_imdb.py>`_.

#     Args:
#         in_channels (int or Dict[str, int]): Size of each input sample of every
#             node type, or :obj:`-1` to derive the size from the first input(s)
#             to the forward method.
#         out_channels (int): Size of each output sample.
#         metadata (Tuple[List[str], List[Tuple[str, str, str]]]): The metadata
#             of the heterogeneous graph, *i.e.* its node and edge types given
#             by a list of strings and a list of string triplets, respectively.
#             See :meth:`torch_geometric.data.HeteroData.metadata` for more
#             information.
#         heads (int, optional): Number of multi-head-attentions.
#             (default: :obj:`1`)
#         negative_slope (float, optional): LeakyReLU angle of the negative
#             slope. (default: :obj:`0.2`)
#         dropout (float, optional): Dropout probability of the normalized
#             attention coefficients which exposes each node to a stochastically
#             sampled neighborhood during training. (default: :obj:`0`)
#         **kwargs (optional): Additional arguments of
#             :class:`torch_geometric.nn.conv.MessagePassing`.
#     """
#     def __init__(
#         self,
#         in_channels: Union[int, Dict[str, int]],
#         out_channels: int,
#         metadata: Metadata,
#         heads: int = 1,
#         negative_slope=0.2,
#         dropout: float = 0.0,
#         **kwargs,
#     ):
#         super().__init__(aggr='add', node_dim=0, **kwargs)

#         if not isinstance(in_channels, dict):
#             in_channels = {node_type: in_channels for node_type in metadata[0]}

#         self.heads = heads
#         self.in_channels = in_channels
#         self.out_channels = out_channels
#         self.negative_slope = negative_slope
#         self.metadata = metadata
#         self.dropout = dropout
#         self.k_lin = nn.Linear(out_channels, out_channels)
#         self.q = nn.Parameter(torch.empty(1, out_channels))

#         self.proj = nn.ModuleDict()
#         for node_type, in_channels in self.in_channels.items():
#             self.proj[node_type] = Linear(in_channels, out_channels)

#         self.lin_src = nn.ParameterDict()
#         self.lin_dst = nn.ParameterDict()
#         dim = out_channels // heads
#         for edge_type in metadata[1]:
#             edge_type = '__'.join(edge_type)
#             self.lin_src[edge_type] = nn.Parameter(torch.empty(1, heads, dim))
#             self.lin_dst[edge_type] = nn.Parameter(torch.empty(1, heads, dim))

#         self.reset_parameters()

#     def reset_parameters(self):
#         super().reset_parameters()
#         reset(self.proj)
#         glorot(self.lin_src)
#         glorot(self.lin_dst)
#         self.k_lin.reset_parameters()
#         glorot(self.q)

#     def forward(
#         self,
#         x_dict: Dict[NodeType, Tensor],
#         edge_index_dict: Dict[EdgeType, Adj],
#         return_semantic_attention_weights: bool = False,
#     ) -> Union[Dict[NodeType, OptTensor], Tuple[Dict[NodeType, OptTensor],
#                                                 Dict[NodeType, OptTensor]]]:
#         r"""Runs the forward pass of the module.

#         Args:
#             x_dict (Dict[str, torch.Tensor]): A dictionary holding node feature
#                 information for each individual node type.
#             edge_index_dict (Dict[Tuple[str, str, str], torch.Tensor]): A
#                 dictionary holding graph connectivity information for each
#                 individual edge type, either as a :class:`torch.Tensor` of
#                 shape :obj:`[2, num_edges]` or a
#                 :class:`torch_sparse.SparseTensor`.
#             return_semantic_attention_weights (bool, optional): If set to
#                 :obj:`True`, will additionally return the semantic-level
#                 attention weights for each destination node type.
#                 (default: :obj:`False`)
#         """
#         H, D = self.heads, self.out_channels // self.heads
#         x_node_dict, out_dict = {}, {}

#         # Iterate over node types:
#         for node_type, x in x_dict.items():
#             x_node_dict[node_type] = self.proj[node_type](x).view(-1, H, D)
#             out_dict[node_type] = []

#         # Iterate over edge types:
#         for edge_type, edge_index in edge_index_dict.items():
#             src_type, _, dst_type = edge_type
#             edge_type = '__'.join(edge_type)
#             lin_src = self.lin_src[edge_type]
#             lin_dst = self.lin_dst[edge_type]
#             x_src = x_node_dict[src_type]
#             x_dst = x_node_dict[dst_type]
#             alpha_src = (x_src * lin_src).sum(dim=-1)
#             alpha_dst = (x_dst * lin_dst).sum(dim=-1)
#             # propagate_type: (x: PairTensor, alpha: PairTensor)
#             out = self.propagate(edge_index, x=(x_src, x_dst),
#                                  alpha=(alpha_src, alpha_dst))

#             out = F.relu(out)
#             out_dict[dst_type].append(out)

#         # iterate over node types:
#         semantic_attn_dict = {}
#         for node_type, outs in out_dict.items():
#             out, attn = group(outs, self.q, self.k_lin)
#             out_dict[node_type] = out
#             semantic_attn_dict[node_type] = attn

#         if return_semantic_attention_weights:
#             return out_dict, semantic_attn_dict

#         return out_dict

#     def message(self, x_j: Tensor, alpha_i: Tensor, alpha_j: Tensor,
#                 index: Tensor, ptr: Optional[Tensor],
#                 size_i: Optional[int]) -> Tensor:

#         alpha = alpha_j + alpha_i
#         alpha = F.leaky_relu(alpha, self.negative_slope)
#         alpha = softmax(alpha, index, ptr, size_i)
#         alpha = F.dropout(alpha, p=self.dropout, training=self.training)
#         out = x_j * alpha.view(-1, self.heads, 1)
#         return out.view(-1, self.out_channels)

#     def __repr__(self) -> str:
#         return (f'{self.__class__.__name__}({self.out_channels}, '
#                 f'heads={self.heads})')


In [4]:
from torch_geometric.datasets import HGBDataset, IMDB

In [6]:
hgb_imdb = IMDB(root='../data/hgb/imdb/')

In [8]:
hgb_imdb[0]

HeteroData(
  movie={
    x=[4278, 3066],
    y=[4278],
    train_mask=[4278],
    val_mask=[4278],
    test_mask=[4278],
  },
  director={ x=[2081, 3066] },
  actor={ x=[5257, 3066] },
  (movie, to, director)={ edge_index=[2, 4278] },
  (movie, to, actor)={ edge_index=[2, 12828] },
  (director, to, movie)={ edge_index=[2, 4278] },
  (actor, to, movie)={ edge_index=[2, 12828] }
)